# Analiza szybkości działania algorytmów (Python)

Ćwiczenie wyjaśniające ideę i techniki empirycznej analizy algorytmów, wraz z wizualizacjami - w Pythonie.

Uruchamiaj komórki kolejno, czytając opisy i wyjaśnienia.

# Inicjalizacja

Przed uruchomieniem jakichkolwiek wizualizacji musimy najpierw zaimportować niezbędne biblioteki i zainicjować je.

- `matplotlib` to biblioteka, która będzie tworzyć i wyświetlać wykresy
- `numpy` to biblioteka zawierająca liczne funkcje matematyczne
- `timeit` to biblioteka, której użyjemy do mierzenia czasu wykonania każdego wywołania algorytmu
- `math` to podstawowa biblioteka matematyczna Pythona
- `random` to podstawowa biblioteka Pythona do generowania wartości losowych

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import timeit
import math
import random

plt.rcParams['figure.figsize'] = [10, 6] # ustawienie rozmiaru i proporcji wykresów

# Wersje demonstracyjne

Poniżej znajdują się fragmenty kodu, które pokazują, jak używać `matplotlib` i `numpy`.

## Funkcja `sum`

Funkcja wbudowana Pythona `sum` oblicza sumę wszystkich elementów dostarczonego obiektu iterowalnego (`Iterable`).

W dokumentacji jest podane, że funkcja ta implementuje algorytm o złożoności czasowej $O(n)$. 

Aby to przetestować, użyjemy metody `linspace` z biblioteki `numpy`, aby wygenerować obiekt iterowalny z 50 równomiernie rozmieszczonymi wartościami w zakresie od 10 do 10 000. Wykres, chociaż nie jest idealnie prostą linią, to obrazuje.

In [ ]:
os_X = np.linspace(10, 10_000, 50, dtype=int) # oś OX
print(os_X)
ts = [timeit.timeit(f'sum(range({n}))', number=100) for n in os_X] # obliczone czasy, oś OY
print(ts)

plt.plot(os_X, ts, 'or')

Możemy dodać linię trendu (używając funkcji 4. stopnia), aby to jeszcze lepiej zilustrować. Wykres nigdy nie będzie idealnie prostą linią, ale powinien być do niej zbliżony.

In [ ]:
degree = 4
coeffs = np.polyfit(os_X, ts, degree)
p = np.poly1d(coeffs)
plt.plot(os_X, ts, 'or')
plt.plot(os_X, [p(n) for n in os_X], '-b')

## Indeksowanie list

Pobieranie elementu z listy (indeksowanie listy) działa ze złożoności czasową $O(1)$, co oznacza, że liczba elementów na liście nie wpływa na czas działania algorytmu. Jak jest to przedstawione na wykresie?

In [ ]:
lst = list(range(1_000_000))
ns = np.linspace(0, len(lst), 1000, endpoint=False, dtype=int)
ts = [timeit.timeit(f'_ = lst[{n}]',
                    globals=globals(), 
                    number=10000) 
      for n in ns]

plt.plot(ns, ts, 'or')

degree = 4
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-b')

# Algorytmy

Teraz przyjrzymy się wykresom generowanym przez następujące algorytmy:

- wyszukiwanie liniowe (_linear search_)
- wyszukiwanie binarne (_binary search_)
- sortowanie przez wstawianie (_insertion sort_)

## Wyszukiwanie liniowe

Wyszukiwanie liniowe ma złożoność czasową $O(n)$, która będzie reprezentowana przez, w przybliżeniu, prostą linię.

**Czerwone punkty** przedstawiają wyszukiwanie elementu w przetasowanej liście, **niebieskie punkty** przedstawiają wyszukiwanie elementu, którego na niej nie ma.

Linia trendu dla czerwonych punktów będzie generalnie niżej niż dla niebieskich punktów, ponieważ wyszukiwanie elementu, którego nie ma na liście, wymaga iteracji przez całą listę.

In [ ]:
## wyszukiwanie elementu w liście
def contains(lst, x):
    for y in lst:
        if x == y: return True
    return False

ns = np.linspace(10, 10_000, 100, dtype=int)

# czerwony wykres
ts = [timeit.timeit('contains(lst, 0)', 
                    setup='lst=list(range({})); random.shuffle(lst)'.format(n),
                    globals=globals(),
                    number=100)
      for n in ns]
plt.plot(ns, ts, 'or')

# linia dopasowania do czerwonego wykesu
degree = 4
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-r')

# niebieski wykres
ts = [timeit.timeit('contains(lst, -1)', 
                    setup='lst=list(range({}))'.format(n),
                    globals=globals(),
                    number=100)
      for n in ns]
plt.plot(ns, ts, 'ob')

# linia dopasowania do niebieskiego wykesu
degree = 4
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-b')

## Wyszukiwanie binarne

Wyszukiwanie binarne działa ze złożonością czasową $O(\log n)$.

Na wykresach czasu działania tej funkcji znajduje się kilka wartości odstających, ale linia trendu bardzo przypomina funkcję logarytmiczną. W idealnej symulacji linia trendu byłaby identyczna z funkcją logarytmiczną.

In [ ]:
# wyszukiwanie elementu w liście wykorzystujące wyszukiwanie binarne
def contains(lst, x):
    lo = 0
    hi = len(lst)-1
    while lo <= hi:
        mid = (lo + hi) // 2
        if x < lst[mid]:
            hi = mid - 1
        elif x > lst[mid]:
            lo = mid + 1
        else:
            return True
    else:
        return False

ns = np.linspace(10, 10000, 1000, dtype=int)
ts = [timeit.timeit('contains(lst, {})'.format(n/2), 
                    setup='lst=list(range({}))'.format(n),
                    globals=globals(),
                    number=1000)
      for n in ns]

plt.plot(ns, ts, 'or')

# dopasowanie wielomianem stopnia 10
degree = 10
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-b')

## Sortowanie przez wstawianie

Jaką złożoność czasową ma sortowanie przez wstawianie? Możemy użyć wykresów czasu jego działania i porównać je z wykresami znanych złożoności, aby określić, które z nich jest do niego najbardziej zbliżone.

In [ ]:
def insertion_sort(lst):
    for i in range(1, len(lst)):
        for j in range(i, 0, -1):
            if lst[j-1] > lst[j]:
                lst[j-1], lst[j] = lst[j], lst[j-1]
            else:
                break

# 15 wartości
ns = np.linspace(100, 2000, 15, dtype=int)
ts = [timeit.timeit('insertion_sort(lst)',
                    setup='lst=list(range({})); random.shuffle(lst)'.format(n),
                    globals=globals(),
                    number=1)
         for n in ns]
plt.plot(ns, ts, 'or');

degree = 4
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-r')

Teraz możemy porównać ten wykres z wykresami różnych złożoności czasowych, aby ostatecznie ustalić, do którego z nich jest najbardziej podobny i jaką złożoność czasową ma sortowanie przez wstawianie.

**Czerwone punkty** reprezentują $y=\log x$, **niebieskie punkty** reprezentują $y=x^2$, **zielone punkty** reprezentują $y=x^x$.

In [ ]:
# y = log x
# vertically stretched 1000x
ns = range(1, 100)
ts = [math.log(n, 2) * 1000 for n in ns]
plt.plot(ns, ts, 'or');

# y = x^2
ns = range(1, 100)
ts = [(n*n) for n in ns]
plt.plot(ns, ts, 'ob');

 
# y = 2^x
# vertically stretched 20x
# horizontally compressed 1x
ns = range(1, 10)
ts = [math.pow(2, n)*20 for n in ns]
plt.plot(ns, ts, 'og');

Na podstawie tych wykresów można bezpiecznie założyć, że sortowanie przez wstawianie działa w czasie $O(n^2)$.

# Analiza czasu działania tajemniczych funkcji

Czy możemy wykorzystać wizualizację czasu działania nieznanych (tajemniczych) funkcji, aby oszacować ich złożoność czasową?

## Tajemnicza funkcja `f`

Jaka jest złożoność obliczeniowa (czasowa) tajemniczej funkcji `f`

- [ ] $O(\log n)$
- [ ] $O(n)$
- [ ] $O(n^3)$
- [ ] $O(n \log n)$

In [ ]:
def f(l: list, val): # l is a python list with n items
  d = {}
  for i in range(len(l)):
    d[l[i]] = i
  return d[val]

ns = range(5, 2000)
ts = [timeit.timeit('f(lst, {})'.format(n-1),
                    setup='lst=list(range({})); random.shuffle(lst)'.format(n),
                    globals=globals(),
                    number=1)
         for n in ns]
plt.plot(ns, ts, 'or');

degree = 4
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-b')

Nawet bez porównywania tego wykresu z wykresami możliwych złożoności czasowych, możemy z góry założyć, że ta funkcja ma złożoność $O(n)$.

## Tajemnicza funkcja `g`

- [ ] $O(n \log n)$
- [ ] $O(n^2)$
- [ ] $O(n)$
- [ ] $O(n^3)$

In [ ]:
def g(l): # l is a python list of integers of length n
  pairs = [ (i,j) for i in range(len(l)) for j in range(len(l)) if i < j ]
  result = []
  for (i,j) in pairs:
    if l[i] == l[j]:
      result.append((i,j))
  return result

ns = range(5, 200)
ts = [timeit.timeit('g(lst)',
                    setup='lst=list(range({}))'.format(n),
                    globals=globals(),
                    number=1)
         for n in ns]
plt.plot(ns, ts, 'or')

degree = 4
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, [p(n) for n in ns], '-b')

Ten wykres wygląda bardzo podobnie do tego dla sortowania przez wstawianie, więc możemy ustalić, że ta funkcja ma złożoność czasową $O(n^2)$.

## Tajemnicza funkcja `h`

Pytanie otwarte - jaka będzie złożoność obliczeniowa?

In [ ]:
def h(n):
   if n <= 1:
       return n
   else:
       return h(n-1) + h(n-2)

ns = range(5, 30)
ts = [timeit.timeit('h({})'.format(n),
                    globals=globals(),
                    number=1)
         for n in ns]
plt.plot(ns, ts, 'or')

Ta funkcja jest bardziej niejednoznaczna niż dwie poprzednie. Widać, że jej złożoność czasowa musi być większa niż $O(n)$, ponieważ nachylenie rośnie wraz ze wzrostem $n$. Rozważmy wykresy złożoności $n^2$ zaznaczone na **czerwono** i $2^n$ na **niebiesko**.

In [ ]:
# y = n^2
# vertically stretched 20x
ns = range(5, 30)
ts = [n*n*20000 for n in ns]
plt.plot(ns, ts, 'or')

# y = 2^n
# vertically compressed 50x
ns = range(5, 30)
ts = [math.pow(2, n)/50 for n in ns]
plt.plot(ns, ts, 'ob')

Wykres czasu działania tajemniczej funkcji `h` bardziej przypomina niebieskie punkty, więc złożoność czasowa tajemniczej funkcji `h` to $O(2^n)$.

# Wnioski

Korzystając z bibliotek do wizualizacji, jesteśmy w stanie określić złożoności czasowe funkcji i algorytmów, porównując je do wykresów znanych złożoności (np. porównując wykresy czasu sortowania przez wstawianie do $y=n^2$). 

Oprócz określania złożoności czasowych, tę metodologię można również wykorzystać do porównywania prędkości różnych algorytmów ze sobą. Za pomocą zaledwie kilku linii kodu możesz szybko sprawdzić, z jaką prędkością będą działać wybrane przez Ciebie algorytmy z dużymi zestawami danych!

# Referencje

- [https://matplotlib.org/stable/tutorials/introductory/pyplot.html](https://matplotlib.org/stable/tutorials/introductory/pyplot.html)
- [https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html)
- [https://www.programiz.com/python-programming/examples/fibonacci-recursion](https://www.programiz.com/python-programming/examples/fibonacci-recursion)